# adversary head の状態 — λ=1 の選択 checkpoint

[adversary_loss_curves.ipynb](adversary_loss_curves.ipynb) で、λ≥1 の adversary loss が
**事前分布だけを答える予測器の loss に張り付く**ことを見た。一方、post-hoc probe では同じ表現から sex が読める。
この食い違いが、次のどちらで起きているのかを checkpoint の重みで切り分ける。

1. **trunk が死んでいる**: `Linear(2048→256) → ReLU` の hidden がほぼ 0 で、head は bias しか使えない。
2. **hidden に情報はあるのに、head がそれを使っていない**: 情報はあるが、head の重みがその方向を向いていない。

対象は λ=1（`20260923T075021Z-…-e41b`）の選択 checkpoint（val AUROC 最大、epoch 10）1 本だけ。
その時点の head の状態を見ているので、他の epoch や他の λ にそのまま広げられない。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import rootutils
import torch
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

from analysis.common.paths import run_dir, split_csv  # noqa: E402
from analysis.common.predictions import features_cache_path, load_features  # noqa: E402
from analysis.common.run_artifacts import selected_checkpoint  # noqa: E402

## 対象 run と定数

特徴量は共有 CLI で作った val の cache（分類 head 直前の 2048 次元、float16 保存）を使う。
adversary の入力は GRL を通した同じ特徴量なので、forward の値は一致する（GRL は勾配だけを変える）。

```bash
uv run python analysis/common/predictions.py --study adversary_strength --split val --features \
  --run-dir analysis/adversary_strength/runs/20260923T075021Z-resnet-chexpert-attribute-invariant-s43-e41b
```

In [ ]:
STUDY = "adversary_strength"
DATASET = "chexpert"
SPLIT = "val"
RUN_ID = "20260923T075021Z-resnet-chexpert-attribute-invariant-s43-e41b"
ADVERSARY_PREFIX = "loss_fn.attribute_adversary."
# categorical_heads の並びは config の adversarial_attribute_names.categorical の順（sex, race）。
CATEGORICAL = ["sex", "race"]
CACHE = ROOT / "analysis" / STUDY / "cache"
RESULTS = ROOT / "analysis" / STUDY / "results"

## 読み込み

checkpoint から adversary の重みだけを取り出し、val の特徴量と split を行を揃えて読む。
age は学習時と同じく train の平均と標準偏差で標準化する。

In [ ]:
checkpoint_path = selected_checkpoint(run_dir(STUDY, RUN_ID))
state = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
adversary = {
    key.removeprefix(ADVERSARY_PREFIX): value.float()
    for key, value in state["state_dict"].items()
    if key.startswith(ADVERSARY_PREFIX)
}

train_split = pd.read_csv(split_csv(DATASET, "train"))
val_split = pd.read_csv(split_csv(DATASET, SPLIT))
features = torch.from_numpy(
    load_features(features_cache_path(CACHE, RUN_ID, SPLIT), val_split["image"].to_numpy())["features"]
)
age_standardized = (val_split["age"] - train_split["age"].mean()) / train_split["age"].std()
checkpoint_path.name, state["epoch"], {key: tuple(value.shape) for key, value in adversary.items()}

## trunk の活性

ReLU の前後の hidden を作り、val の全行で一度も正にならない unit（死んだ unit）を数える。

In [ ]:
pre_activation = features @ adversary["trunk.0.weight"].T + adversary["trunk.0.bias"]
hidden = pre_activation.clamp(min=0)
active_share = (hidden > 0).float().mean(dim=0)

pd.Series(
    {
        "units": hidden.shape[1],
        "dead_units (val で一度も発火しない)": int((active_share == 0).sum()),
        "units_active_over_1pct": int((active_share > 0.01).sum()),
        "mean_active_units_per_sample": float((hidden > 0).float().sum(dim=1).mean()),
    }
)

## head の出力

head の出力が特徴で変わっているか（行ごとのばらつき）と、属性を並べ替えられているか（sex の AUROC）を見る。
loss は学習時と同じく、属性が観測された行だけで計算する。

In [ ]:
head_rows = []
for index, name in enumerate(CATEGORICAL):
    logits = hidden @ adversary[f"categorical_heads.{index}.weight"].T + adversary[f"categorical_heads.{index}.bias"]
    observed = ~val_split[f"{name}_missing"].astype(bool).to_numpy()
    labels = torch.from_numpy(val_split[name].to_numpy()[observed]).long()
    head_rows.append(
        {
            "attribute": name,
            "cross_entropy": float(F.cross_entropy(logits[observed], labels)),
            "max_logit_std_over_rows": float(logits.std(dim=0).max()),
            "head_weight_norm": float(adversary[f"categorical_heads.{index}.weight"].norm()),
        }
    )
    if name == "sex":
        sex_logits = logits

sex_score = (sex_logits[:, 1] - sex_logits[:, 0]).numpy()
sex_probability = torch.softmax(sex_logits, dim=1)[:, 1].numpy()
age_prediction = (hidden @ adversary["continuous_head.weight"].T + adversary["continuous_head.bias"]).squeeze(1).numpy()

display(pd.DataFrame(head_rows).round(4))
pd.Series(
    {
        "sex head AUROC": roc_auc_score(val_split["sex"], sex_score),
        "P(sex=1) min": sex_probability.min(),
        "P(sex=1) max": sex_probability.max(),
        "share predicted sex=1": (sex_score > 0).mean(),
        "age MSE": float(((age_prediction - age_standardized) ** 2).mean()),
        "age prediction std": age_prediction.std(),
        "age corr": np.corrcoef(age_prediction, age_standardized)[0, 1],
    }
).round(4)

## hidden に sex の情報は残っているか

同じ val を患者単位で半分に割り、片方で線形 probe（logistic 回帰、class balanced）を fit して、もう片方で評価する。
3 つの段で比べる: backbone の特徴量（adversary の入力）、trunk の ReLU 前、ReLU 後（head の入力）。
**ReLU 後で読めるのに head が chance なら、情報はあるのに head が使っていない。**

val だけで fit しているので、post-hoc probe（train で fit）の数値とは直接比べない。段の間の比較だけに使う。
2048 次元の fit に数分かかる。

In [ ]:
patients = val_split["image"].str.extract(r"(patient\d+)")[0]
fit_rows, evaluate_rows = next(
    GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=0).split(features, groups=patients)
)
sex = val_split["sex"].to_numpy()

representations = {
    "backbone feature (2048)": features.numpy(),
    "trunk pre-ReLU (256)": pre_activation.numpy(),
    "trunk post-ReLU (256)": hidden.numpy(),
}
probe_rows = []
for name, values in representations.items():
    scaler = StandardScaler().fit(values[fit_rows])
    probe = LogisticRegression(max_iter=2000, class_weight="balanced").fit(
        scaler.transform(values[fit_rows]), sex[fit_rows]
    )
    scores = probe.decision_function(scaler.transform(values[evaluate_rows]))
    probe_rows.append(
        {
            "representation": name,
            "sex_bacc": balanced_accuracy_score(sex[evaluate_rows], scores > 0),
            "sex_auroc": roc_auc_score(sex[evaluate_rows], scores),
        }
    )
probe_stages = pd.DataFrame(probe_rows)
probe_stages.to_csv(RESULTS / "adversary_head_sex_probe_val.csv", index=False)
probe_stages.round(4)

## まとめ

λ=1 の選択 checkpoint（epoch 10）、val で見た結果。

- **trunk は一部死んでいるが、全部ではない。** 256 unit のうち 206 は val で一度も発火しない。1% 以上の行で発火する unit は 42 で、1 行あたり平均 29 unit が発火する。
- **sex head は特徴を使っていない。** AUROC 0.4998、P(sex=1) は 0.385〜0.569 の狭い範囲にあり、sex=1 と答える行は 0.08%。cross-entropy 0.6742 は事前分布だけを答えた loss（0.6737）とほぼ同じ。
- **それでも head の入力には sex の情報が残っている。** 線形 probe の sex BAcc は、backbone の特徴量 0.856 → trunk の ReLU 前 0.783 → ReLU 後 0.681 と段ごとに落ちるが、ReLU 後でも chance（0.5）より明らかに高い。
- age head も予測の標準偏差が 0.13（標準化後）しかなく、相関 0.26、MSE 0.933 で、ほとんど読めていない。

**解釈候補**: 1（trunk の全滅）ではなく、2 に近い。head の入力に線形で読める sex の方向があるのに、head の重みはその方向を向いていない。
GRL で backbone が head の今の重みの方向から sex を逃がし、head がそれに追いつけない、という minimax の失敗が考えられる。
死んだ 206 unit は、head が使える情報をさらに減らしている（ReLU 前後で BAcc 0.783 → 0.681）。

seed 1 本、λ=1、1 checkpoint だけの観察であり、他の λ と epoch はまだ見ていない。